In [1]:
import numpy as np
import h5py
import os
from utilz.preprocessing_utilz import load_stim_metadata
from skimage import io, color, transform, filters
from sklearn.decomposition import PCA
import plenoptic as po
import torch

/home/nirajesh/erp_interp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load img_from_metadata
img_metadata_dir = "../data/erp_processed/config2/sub-01_train_stim_metadata.h5"
img_metadata = load_stim_metadata(img_metadata_dir)
stim_ids = img_metadata["stim_ids"]
stim_files = img_metadata["img_files"]


  Loaded stimulus metadata from ../data/erp_processed/config2/sub-01_train_stim_metadata.h5 for subject 1 split train


In [53]:
stim_dir = '../data/THINGS_images/images/object_images'

import re
def load_img(img_path: str, crop_size: int =224, grayscale: bool =True) -> np.ndarray:
    # object name is the first part of the img path before the number
    object_name = re.split(r'_\d+', img_path)[0]
    if object_name.endswith('_'):
        object_name = object_name[:-1]

    stim_path = os.path.join(stim_dir, object_name, img_path)
    img = io.imread(stim_path)
    if img.ndim == 2:
        img_gray = img.astype(np.float32)
        if img_gray.max() > 1:
            img_gray = img_gray / 255.0
        return img_gray
    
    img = img[..., :3]
    if img.max() > 1:
        img = img.astype(np.float32) / 255.0
    else:
        img = img.astype(np.float32)

    if grayscale:
        img = color.rgb2gray(img)

    img = transform.resize(img, (crop_size, crop_size), anti_aliasing=True)
    return img.astype(np.float32)


def get_pixel_features(img_gray: np.ndarray) -> np.ndarray:
    img_flat = img_gray.ravel()
    return img_flat.astype(np.float32)


In [46]:
def compute_gradients(img_gray: np.ndarray, sigma: float =1.0) -> tuple[np.ndarray, np.ndarray]:
    smoothed = filters.gaussian(img_gray, sigma=sigma)
    gx = filters.sobel_h(smoothed)
    gy = filters.sobel_v(smoothed)
    magnitude = np.sqrt(gx ** 2 + gy ** 2)
    orientation = np.mod(np.arctan2(gy, gx), np.pi)
    return magnitude, orientation


def orientation_histogram(magnitude: np.ndarray, orientation: np.ndarray, n_bins: int =8) -> np.ndarray:
    bins = np.linspace(0, np.pi, n_bins + 1)
    hist, _ = np.histogram(orientation.ravel(), bins=bins, weights=magnitude.ravel())
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    return hist


def get_low_level_features(
    img_gray: np.ndarray,
    sigma: float =1.0,
    n_bins: int =8,
    grid_size: tuple[int, int] = (4, 4),
    edge_percentile: int =75,
) -> np.ndarray:
    magnitude, orientation = compute_gradients(img_gray, sigma=sigma)

    feats = []

    thr = np.percentile(magnitude, edge_percentile)
    edge_mask = magnitude >= thr

    feats.extend([
        float(magnitude.mean()),
        float(magnitude.std()),
        float(magnitude.max()),
        float(edge_mask.mean()),
    ])

    global_hist = orientation_histogram(magnitude, orientation, n_bins=n_bins)
    feats.extend(global_hist.tolist())

    gh, gw = grid_size
    h, w = img_gray.shape
    cell_h = h // gh
    cell_w = w // gw

    for i in range(gh):
        for j in range(gw):
            r0, r1 = i * cell_h, (i + 1) * cell_h if i < gh - 1 else h
            c0, c1 = j * cell_w, (j + 1) * cell_w if j < gw - 1 else w

            mag_cell = magnitude[r0:r1, c0:c1]
            ori_cell = orientation[r0:r1, c0:c1]
            hist_cell = orientation_histogram(mag_cell, ori_cell, n_bins=n_bins)
            feats.extend(hist_cell.tolist())

    return np.array(feats, dtype=np.float32)

In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_midlevel_features(img: np.ndarray) -> np.ndarray:
    po.tools.set_seed(42)
    assert img.ndim == 4, "Input must be a batched images with shape (N, C, H, W)"
    model = po.simul.PortillaSimoncelli(
        image_shape = (224, 224),
        n_scales = 4,
        n_orientations = 4,
        spatial_corr_width=10
    )
    features = model(img)
    #return one of the channels
    out = features[:, 0, :].cpu().numpy().astype(np.float32)
    return out

In [59]:
from tqdm import tqdm
num_stims = len(stim_files)
max_feature_dim = 100

pixel_features = []
lowlevel_features = []
midlevel_features = []

for stim_i, stim_file in enumerate(tqdm(stim_files[:200])):
    img = load_img(stim_file, grayscale=True)
    pixel_features.append(get_pixel_features(img))

    lowlevel_features.append(get_low_level_features(img))

    img_3dims = np.stack([img, img, img], axis=-1)
    img_batched = np.expand_dims(img_3dims, axis=0).transpose(0, 3, 1, 2)  # (N, C, H, W)
    midlevel_features.append(get_midlevel_features(torch.tensor(img_batched, dtype=torch.float32).to(device)))

print("Extracted features for all stims, now applying PCA if needed...")



100%|██████████| 200/200 [01:34<00:00,  2.12it/s]

Extracted features for all stims, now applying PCA if needed...


In [63]:

pixel_features = np.array(pixel_features, dtype=np.float32)
if pixel_features.shape[1] > max_feature_dim:
    print(f"Pixel features have {pixel_features.shape[1]} dimensions, applying PCA to reduce to {max_feature_dim}...")
    pca = PCA(n_components=max_feature_dim)
    pixel_features = pca.fit_transform(pixel_features)
print("Pixel features shape:", pixel_features.shape)


lowlevel_features = np.array(lowlevel_features, dtype=np.float32)
if lowlevel_features.shape[1] > max_feature_dim:
    print(f"Low-level features have {lowlevel_features.shape[1]} dimensions, applying PCA to reduce to {max_feature_dim}...")
    pca = PCA(n_components=max_feature_dim)
    lowlevel_features = pca.fit_transform(lowlevel_features)
print("Low-level features shape:", lowlevel_features.shape)

# midlevel_features = np.array(midlevel_features[:,0,:], dtype=np.float32)
midlevel_features = np.array(midlevel_features, dtype=np.float32)
if midlevel_features.shape[1] > max_feature_dim:
    print(f"Mid-level features have {midlevel_features.shape[1]} dimensions, applying PCA to reduce to {max_feature_dim}...")
    pca = PCA(n_components=max_feature_dim)
    midlevel_features = pca.fit_transform(midlevel_features)
print("Mid-level features shape:", midlevel_features.shape)

Pixel features shape: (200, 100)
Low-level features shape: (200, 100)
Mid-level features shape: (200, 100)


In [64]:
# save each feature bank as a npz file
outdir = "./"

np.savez(os.path.join(outdir, "pixel_features.npz"), features=pixel_features)
np.savez(os.path.join(outdir, "lowlevel_features.npz"), features=lowlevel_features)
np.savez(os.path.join(outdir, "midlevel_features.npz"), features=midlevel_features)


## DNN feats

In [3]:
import numpy as np
import h5py
import os
from utilz.preprocessing_utilz import load_stim_metadata
from skimage import io, color, transform, filters
from sklearn.decomposition import PCA
import plenoptic as po
import torch



# load img_from_metadata
img_metadata_dir = "../data/erp_processed/config2/sub-01_train_stim_metadata.h5"
img_metadata = load_stim_metadata(img_metadata_dir)
stim_ids = img_metadata["stim_ids"]
stim_files = img_metadata["img_files"]
stim_dir = '../data/THINGS_images/images/object_images'



  Loaded stimulus metadata from ../data/erp_processed/config2/sub-01_train_stim_metadata.h5 for subject 1 split train


In [ ]:
from utilz.dnns import make_things_dataloader, load_model, extract_model_activations

model, preprocess = load_model("vgg16")
dl = make_things_dataloader(stim_dir, stim_files[:200], preprocess, batch_size=32, num_workers=4)

print(f"Loaded dataloader with {len(dl.dataset)} images, ready to extract features using VGG16...")

Loaded model: vgg16_bn.tv_in1k with pretrained=True. Found preprocessing function: True
Loaded dataloader with 16540 images, ready to extract features using VGG16...


In [ ]:
device = torch.device(f'cuda:2' if torch.cuda.is_available() else 'cpu')
acts = extract_model_activations(model, region='early', dataloader=dl, device=device)
extracted_layers = list(acts.keys())
print(f"Extracted activations from layers: {extracted_layers}")

In [21]:
acts['features.0'].shape

(16540, 64)